# 파싱 검증 파이프라인 실행 노트북

GT(.docx/.pptx/.xlsx) vs 파서 결과(before.txt / after.txt) 비교.
- docx/pptx → **WER**(단어) + **TEDS-content**(표)
- xlsx → **TEDS-content** (null 패딩 등 제거하는 엑셀 전용 전처리 후, `xlsx_eval.py`)

텍스트 지표는 WER 하나만 쓰고, 띄어쓰기 밀림 등 상황은 `config.EVAL_PROFILE`
(전처리 프로파일)로 대응한다. 케이스 비교는 `run_eval.compare_profiles(doc_id)`.

실행 전: 이 노트북과 `.py` 모듈들, `data/` 폴더가 같은 폴더에 있어야 한다.

## 1. (최초 1회) 패키지 설치

In [6]:
!pip install -r requirements.txt

## 2. 문서셋 전체 평가
모든 문서를 돌며 WER/TEDS를 계산하고 before→after 개선폭을 요약한다.
결과 CSV는 `output/` 폴더에 저장된다.

In [7]:
import run_eval
df, summary = run_eval.run()

문서별 상세 결과 (전처리 프로파일: default, 토큰: eojeol)
                doc_id type  dual_gt  wer_before  teds_before  wer_after  teds_after
            ppt_slides pptx     True    0.419355          NaN   0.142857         1.0
  word_box_memo_strike docx     True    0.620000          NaN   0.028571         1.0
       word_text_table docx     True    0.166667     0.884615   0.000000         1.0
xlsx_multisheet_linked xlsx     True         NaN     1.000000        NaN         1.0
           xlsx_single xlsx     True         NaN     0.619048        NaN         1.0

문서셋 전체 요약 (Before → After)
- wer          : 0.402 → 0.0571  (감소(개선) 85.79%)
- teds_content : 0.8346 → 1.0  (증가(개선) 19.82%)

[상세 CSV] output/report_per_document.csv
[요약 CSV] output/report_summary.csv


## 3. 문서별 상세 표 보기 (선택)

In [ ]:
df   # 문서별 WER/정규화WER/TEDS 상세

## 4. 특정 문서 진단 (선택)
한 문서가 왜 그런 점수인지 단어/문단/표 단위로 자세히 본다.
`stage`는 `"before"` 또는 `"after"`. `make_html=True`면 3-way HTML 리포트도 만든다.

In [7]:
run_eval.diagnose("xlsx_multisheet_linked", stage="before", make_html=True)  # 한 문서 상세

[오류] xlsx_multisheet_linked의 GT(_gt 또는 _before_gt)를 찾을 수 없습니다.


In [3]:
run_eval.diagnose("ppt_slides", stage="before", make_html=True)  # 한 문서 상세
run_eval.diagnose("word_box_memo_strike", stage="before", make_html=True)  # 한 문서 상세
run_eval.diagnose("word_text_table", stage="before", make_html=True)  # 한 문서 상세
run_eval.diagnose("xlsx_multisheet_linked", stage="before", make_html=True)  # 한 문서 상세
run_eval.diagnose("xlsx_single", stage="before", make_html=True)  # 한 문서 상세

[ppt_slides / before] 요약  (프로파일 default)  WER 0.3871  |  TEDS-content None

[문단 단위 diff] (큰 그림: 어디서 갈라지는지)
[GT≠]   JURIS 준법심의 절차
[PRED≠] ## JURIS 준법심의 절차
[=] 심의는 접수 후 진행한다
[GT≠]   예외: 긴급 광고물은 사전 승인 없이 사후 심의 가능
[PRED≠] ## 특별점검 절차
[GT≠]   특별점검 절차
[=] 정기점검과 특별점검으로 나뉜다
[GT≠]   현장점검은 실시한다
[PRED≠] - 현장점검은 실시한다
[GT≠]   특별점검 절차
[PRED≠] ## 특별점검 절차
[=] 구분
[=] 1월
[=] 2월
[=] 저축은행
[=] 429
[=] 322

[단어 단위 diff] (어떤 단어가 문제인지)
WER=0.3871  S=1 D=8 I=3 / N=31
------------------------------------------------------------
[삽입 I]  GT: (없음)   PRED: ##
[누락 D]  GT: 예외:   PRED: (없음)
[누락 D]  GT: 긴급   PRED: (없음)
[누락 D]  GT: 광고물은   PRED: (없음)
[누락 D]  GT: 사전   PRED: (없음)
[누락 D]  GT: 승인   PRED: (없음)
[누락 D]  GT: 없이   PRED: (없음)
[누락 D]  GT: 사후   PRED: (없음)
[누락 D]  GT: 심의   PRED: (없음)
[치환 S]  GT: 가능   PRED: ##
[삽입 I]  GT: (없음)   PRED: -
[삽입 I]  GT: (없음)   PRED: ##
[HTML 리포트] output/report_ppt_slides.html
[word_box_memo_strike / before] 요약  (프로파일 default)  WER 0.6200  |  TEDS-content None

[문단 단위 diff] (큰 그림: 어디서 갈라지는지)

## 5. 전처리 케이스 비교 (선택)
한 문서가 왜 점수가 나쁜지(띄어쓰기? 서식? 내용?) 프로파일별 WER로 확인.

In [ ]:
run_eval.compare_profiles(ids[0], stage="before")